# Fix Transaction Datetime Hour Information (Dataframe Only)

Fixes missing hour information in dataframes by adding "00:00:00".

In [2]:
import pandas as pd

csv_files = ['wire.csv', 'cheque.csv']

In [ ]:
def fix_datetime_hour(filename):
    """Fix missing hour information by adding 00:00:00 to date-only values. Returns modified dataframe (does NOT save to CSV)."""
    df = pd.read_csv(filename)
    date_only = df['transaction_datetime'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
    rows_to_fix = date_only.sum()
    df.head()
    if rows_to_fix > 0:
        df.loc[date_only, 'transaction_datetime'] = df.loc[date_only, 'transaction_datetime'] + ' 00:00:00'
        return {'fixed': True, 'rows_fixed': rows_to_fix, 'df': df}
    return {'fixed': False, 'df': df}

In [ ]:
# Fix dataframes
fixed_dataframes = {}

for csv_file in csv_files:
    result = fix_datetime_hour(csv_file)
    if result['fixed']:
        print(f"{csv_file}: Fixed {result['rows_fixed']:,} rows in dataframe (CSV not modified)")
        fixed_dataframes[csv_file] = result['df']
    else:
        print(f"{csv_file}: Already has hour information")
        fixed_dataframes[csv_file] = result['df']

wire.csv: Fixed 4,956 rows in dataframe (CSV not modified)
cheque.csv: Fixed 240,548 rows in dataframe (CSV not modified)


In [ ]:
# Verify fixed dataframes
print("\nVerification (checking fixed dataframes in memory):")
for csv_file, df in fixed_dataframes.items():
    date_only = df['transaction_datetime'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
    if date_only.sum() == 0:
        print(f"✓ {csv_file}: All rows have proper datetime format in dataframe")
    else:
        print(f"✗ {csv_file}: {date_only.sum():,} rows still missing hour in dataframe")


Verification (checking fixed dataframes in memory):
✓ wire.csv: All rows have proper datetime format in dataframe
✓ cheque.csv: All rows have proper datetime format in dataframe


In [6]:
# Save fixed dataframes to new CSV files
output_files = {
    'wire.csv': 'Cleaned_wire.csv',
    'cheque.csv': 'Cleaned_cheque.csv'
}

for original_file, output_file in output_files.items():
    if original_file in fixed_dataframes:
        fixed_dataframes[original_file].to_csv(output_file, index=False)
        print(f"✓ Saved {output_file} ({len(fixed_dataframes[original_file]):,} rows)")

✓ Saved Cleaned_wire.csv (4,956 rows)
✓ Saved Cleaned_cheque.csv (240,548 rows)
